# 🔬 5주차 보강 — 시각화와 EDA, "왜?"를 묻다

> **그래프를 예쁘게 그리는 법이 아니라, 왜 시각화해야 하는지, 그것이 AI와 어떻게 연결되는지를 이해한다.**

---

## 이 노트북의 목표

| 기존 (도구 설명서 수준) | 보강 (전문가 수준 + 최신 AI 연결) |
|---|---|
| "`sns.histplot()`으로 히스토그램을 그린다" | 분포를 **왜** 봐야 하는가? → **정규분포 가정** → ML 알고리즘의 전제 조건 |
| "`sns.boxplot()`으로 박스플롯을 그린다" | **이상치**(Outlier)가 **왜** 중요한가? → 모델 성능 저하 → **강건한 AI**(Robust AI) |
| "`sns.heatmap()`으로 상관관계를 본다" | 상관관계가 **왜** 중요한가? → **특성 선택**(Feature Selection) → 차원의 저주 |
| "EDA를 하면 데이터를 이해할 수 있다" | EDA가 **왜** ML 파이프라인의 필수인가? → **데이터 중심 AI** → AutoML도 EDA부터 시작 |
| "`plt.show()`로 그래프를 표시한다" | 시각화가 **왜** 인간 고유의 능력인가? → **설명 가능한 AI**(XAI) → 모델 해석 |

### 🔑 핵심 메시지

> *"시각화는 '예쁜 그래프'를 만드는 기술이 아니다. 데이터에 숨겨진 패턴, 문제, 가능성을 인간의 눈으로 발견하는 기술이다. GPT는 텍스트를 생성하지만, 데이터를 '보는' 것은 아직 인간의 영역이다."*


---

## 1. "왜 시각화해야 하는가?" — 숫자는 거짓말한다

### 앤스콤의 사중주(Anscombe's Quartet)를 넘어서

4주차에서 `describe()`로 통계량을 봤다. 그런데 **통계량이 같아도 데이터가 완전히 다를 수 있다.**

이것을 처음 보여준 것이 **앤스콤의 사중주**(1973)이고, 이를 극적으로 확장한 것이 **데이터사우루스**(Datasaurus Dozen, 2017)이다.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import pandas as pd
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

# ── 한글 폰트 ──
if 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    os.system('apt-get -qq install -y fonts-nanum > /dev/null 2>&1')
    import matplotlib.font_manager as fm
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if os.path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100
sns.set_palette("colorblind")

C = {
    'input': '#3b82f6', 'weight': '#f59e0b', 'bias': '#8b5cf6',
    'sum': '#ef4444', 'act': '#10b981', 'output': '#10b981',
    'bg': '#0f172a', 'card': '#1e293b', 'text': '#e2e8f0',
    'dim': '#94a3b8', 'muted': '#64748b', 'accent': '#60a5fa',
}

# ═══════════════════════════════════════════════════
# 앤스콤의 사중주 — "통계량은 같지만 데이터는 완전히 다르다"
# ═══════════════════════════════════════════════════

anscombe = sns.load_dataset('anscombe')

fig, axes = plt.subplots(2, 4, figsize=(16, 8), facecolor=C['bg'])

colors = [C['input'], C['act'], C['weight'], C['sum']]
datasets = ['I', 'II', 'III', 'IV']

for i, (ds, color) in enumerate(zip(datasets, colors)):
    subset = anscombe[anscombe['dataset'] == ds]

    # 상단: 통계량
    ax_stat = axes[0, i]
    ax_stat.set_facecolor(C['card'])
    ax_stat.axis('off')
    ax_stat.set_title(f'데이터셋 {ds}', color=color, fontsize=13, fontweight='bold')

    stats = [
        f"mean(x) = {subset['x'].mean():.1f}",
        f"mean(y) = {subset['y'].mean():.2f}",
        f"std(x)  = {subset['x'].std():.2f}",
        f"std(y)  = {subset['y'].std():.2f}",
        f"corr    = {subset['x'].corr(subset['y']):.3f}",
    ]
    for j, stat in enumerate(stats):
        ax_stat.text(0.5, 0.85 - j*0.18, stat, transform=ax_stat.transAxes,
                     ha='center', va='center', color=C['text'], fontsize=10,
                     fontfamily='monospace')

    # 하단: 산점도
    ax_plot = axes[1, i]
    ax_plot.set_facecolor(C['card'])
    ax_plot.scatter(subset['x'], subset['y'], c=color, s=50, alpha=0.7,
                    edgecolors='white', linewidths=0.5)

    # 회귀선
    m, b = np.polyfit(subset['x'], subset['y'], 1)
    x_line = np.linspace(subset['x'].min()-1, subset['x'].max()+1, 100)
    ax_plot.plot(x_line, m*x_line + b, color=color, lw=2, alpha=0.5, ls='--')

    ax_plot.tick_params(colors=C['muted'], labelsize=8)
    for spine in ax_plot.spines.values():
        spine.set_color(C['muted'])
        spine.set_alpha(0.3)

# 가운데 메시지
axes[0, 0].text(-0.3, 0.5, '통계량\n(숫자)', transform=axes[0,0].transAxes,
                ha='center', va='center', color=C['dim'], fontsize=11,
                fontweight='bold', rotation=90)
axes[1, 0].text(-0.3, 0.5, '시각화\n(그래프)', transform=axes[1,0].transAxes,
                ha='center', va='center', color=C['accent'], fontsize=11,
                fontweight='bold', rotation=90)

plt.suptitle('앤스콤의 사중주(Anscombe\'s Quartet) — 통계량은 동일, 패턴은 완전히 다르다',
             color=C['text'], fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 교훈: 통계량(mean, std, corr)만으로는 데이터를 이해할 수 없다.")
print("   → '항상 그래프를 먼저 그려라' — 이것이 EDA의 제1원칙이다.")
print("\n🔗 AI 연결:")
print("   ML 모델도 숫자(손실 함수 값)만 보면 속는다.")
print("   학습 곡선(Learning Curve), 잔차 플롯(Residual Plot)을 시각화해야")
print("   과적합, 데이터 누수(Data Leakage) 같은 문제를 발견할 수 있다.")


---

## 2. 분포(Distribution)를 보는 이유 — ML 알고리즘의 전제 조건

### 왜 히스토그램을 그려야 하는가?

"데이터 분포를 확인하라"는 것은 단순한 습관이 아니다. **많은 ML 알고리즘이 정규분포를 가정**하기 때문이다.

| 알고리즘 | 정규분포 가정 | 위반 시 문제 |
|---|---|---|
| **선형 회귀** | 잔차(Residual)가 정규분포 | 신뢰구간 부정확, p-value 무의미 |
| **로지스틱 회귀** | 특성이 대략 정규분포 | 극단값에 민감, 성능 저하 |
| **KNN** | 특성 스케일이 유사 | 스케일 큰 특성이 거리를 지배 |
| **SVM** | 특성이 스케일링됨 | 서포트 벡터 선택 오류 |
| **신경망** | 입력이 정규화됨 | 학습 속도 극감, 기울기 폭발 |

> **🔗 AI 연결**: GPT/BERT도 입력 **임베딩을 정규화**(Layer Normalization)한다. "분포를 정규화하라"는 원칙은 전통 ML부터 최신 LLM까지 관통한다.


In [ ]:
# ═══════════════════════════════════════════════════
# 분포가 ML에 미치는 영향 시각화
# ═══════════════════════════════════════════════════
from scipy import stats

np.random.seed(42)

fig, axes = plt.subplots(2, 3, figsize=(16, 10), facecolor=C['bg'])

# ── 상단: 3가지 분포 유형 ──
distributions = [
    ("정규분포 (이상적)", np.random.normal(50, 10, 1000), C['act']),
    ("왼쪽 치우침 (Skewed)", np.random.exponential(10, 1000) + 20, C['weight']),
    ("이봉분포 (Bimodal)", np.concatenate([np.random.normal(30, 5, 500),
                                           np.random.normal(70, 5, 500)]), C['bias']),
]

for i, (title, data, color) in enumerate(distributions):
    ax = axes[0, i]
    ax.set_facecolor(C['card'])
    ax.hist(data, bins=40, color=color, alpha=0.5, edgecolor='white', lw=0.5,
            density=True)
    ax.axvline(np.mean(data), color=C['sum'], lw=2, ls='--', label=f'평균={np.mean(data):.1f}')
    ax.axvline(np.median(data), color=C['accent'], lw=2, ls=':', label=f'중앙값={np.median(data):.1f}')
    ax.set_title(title, color=color, fontsize=13, fontweight='bold')
    ax.legend(fontsize=8, facecolor=C['bg'], edgecolor=C['muted'], labelcolor=C['dim'])
    ax.tick_params(colors=C['muted'], labelsize=8)
    for spine in ax.spines.values():
        spine.set_color(C['muted'])
        spine.set_alpha(0.3)

# ── 하단: 정규화 전후 비교 ──
# 왜도 큰 데이터
skewed_data = np.random.exponential(10, 1000)

# 로그 변환
log_data = np.log1p(skewed_data)

# 표준화(StandardScaler)
from sklearn.preprocessing import StandardScaler
scaled_data = StandardScaler().fit_transform(skewed_data.reshape(-1, 1)).flatten()

transforms = [
    ("원본 (왜도 큰 분포)", skewed_data, C['sum'],
     f"왜도(skew)={stats.skew(skewed_data):.2f}"),
    ("로그 변환 log(1+x)", log_data, C['act'],
     f"왜도(skew)={stats.skew(log_data):.2f}"),
    ("표준화 (StandardScaler)", scaled_data, C['accent'],
     f"평균≈0, 표준편차≈1"),
]

for i, (title, data, color, annotation) in enumerate(transforms):
    ax = axes[1, i]
    ax.set_facecolor(C['card'])
    ax.hist(data, bins=40, color=color, alpha=0.5, edgecolor='white', lw=0.5,
            density=True)
    ax.set_title(title, color=color, fontsize=13, fontweight='bold')
    ax.text(0.95, 0.92, annotation, transform=ax.transAxes,
            ha='right', va='top', color=color, fontsize=10, fontweight='bold',
            fontfamily='monospace',
            bbox=dict(facecolor=C['bg'], edgecolor=color,
                     boxstyle='round,pad=0.3', alpha=0.9))
    ax.tick_params(colors=C['muted'], labelsize=8)
    for spine in ax.spines.values():
        spine.set_color(C['muted'])
        spine.set_alpha(0.3)

plt.suptitle('분포(Distribution)가 중요한 이유 — ML은 정규분포를 좋아한다',
             color=C['text'], fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n🐍 핵심 코드:")
print("   # 왜도 확인")
print("   from scipy.stats import skew")
print("   print(f'왜도: {skew(data):.2f}')   # |왜도| > 1이면 변환 고려")
print("   ")
print("   # 로그 변환 (왜도 줄이기)")
print("   data_log = np.log1p(data)")
print("   ")
print("   # 표준화 (평균=0, 표준편차=1)")
print("   from sklearn.preprocessing import StandardScaler")
print("   data_scaled = StandardScaler().fit_transform(data)")
print("\n🔗 AI 연결:")
print("   GPT/BERT의 Layer Normalization: 각 층의 출력을 평균=0, 분산=1로 정규화")
print("   원리는 StandardScaler와 동일 → '분포를 정규화하면 학습이 안정된다'")


---

## 3. 이상치(Outlier) — "왜 제거해야 하는가? 항상 제거해야 하는가?"

### 이상치는 항상 나쁜가?

**아니다.** 이상치에는 두 종류가 있다:

| 유형 | 설명 | 처리 |
|---|---|---|
| **오류**(Error) | 입력 실수, 센서 고장, 단위 착오 | **제거** 또는 수정 |
| **희소 사건**(Rare Event) | 사기 거래, 이상 탐지 | **보존** — 이것이 핵심 정보! |

> **🔗 AI 연결**: 신용카드 **사기 탐지**(Fraud Detection)에서는 이상치가 곧 "사기"이다. 이상치를 제거하면 탐지 모델이 무용지물이 된다. **이상 탐지**(Anomaly Detection)는 이상치를 "찾는" 것이 목표인 AI 분야이다.


In [ ]:
# ═══════════════════════════════════════════════════
# 이상치가 모델에 미치는 영향 시각화
# ═══════════════════════════════════════════════════

np.random.seed(42)
n = 100
x = np.random.uniform(10, 90, n)
y = 0.5 * x + np.random.normal(0, 5, n) + 20

# 이상치 추가
x_outlier = np.append(x, [95, 92, 88])
y_outlier = np.append(y, [10, 5, 15])  # 극단적 이상치

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=C['bg'])

# ── 이상치 없음 ──
ax = axes[0]
ax.set_facecolor(C['card'])
ax.scatter(x, y, c=C['input'], s=30, alpha=0.5, edgecolors='white', lw=0.5)
m, b_val = np.polyfit(x, y, 1)
ax.plot(x, m*x + b_val, color=C['act'], lw=2, label=f'y = {m:.2f}x + {b_val:.1f}')
ax.set_title('① 이상치 없음 (정상)', color=C['act'], fontsize=13, fontweight='bold')
ax.legend(fontsize=9, facecolor=C['bg'], edgecolor=C['muted'], labelcolor=C['dim'])
ax.tick_params(colors=C['muted'])
for spine in ax.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

# ── 이상치 있음 ──
ax = axes[1]
ax.set_facecolor(C['card'])
ax.scatter(x, y, c=C['input'], s=30, alpha=0.5, edgecolors='white', lw=0.5)
ax.scatter([95, 92, 88], [10, 5, 15], c=C['sum'], s=80, marker='X',
           edgecolors='white', lw=1, zorder=5, label='이상치')
m2, b2 = np.polyfit(x_outlier, y_outlier, 1)
ax.plot(x_outlier, m2*x_outlier + b2, color=C['sum'], lw=2,
        label=f'y = {m2:.2f}x + {b2:.1f}')
# 원래 선도 표시
ax.plot(x, m*x + b_val, color=C['act'], lw=1, ls='--', alpha=0.5, label='원래 선')
ax.set_title('② 이상치 3개 추가 → 회귀선 왜곡!', color=C['sum'],
             fontsize=13, fontweight='bold')
ax.legend(fontsize=8, facecolor=C['bg'], edgecolor=C['muted'], labelcolor=C['dim'])
ax.tick_params(colors=C['muted'])
for spine in ax.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

# ── 이상치 = 핵심 정보 ──
ax = axes[2]
ax.set_facecolor(C['card'])

# 정상 거래 vs 사기 거래
normal_x = np.random.normal(50, 15, 200)
normal_y = np.random.normal(50, 15, 200)
fraud_x = np.random.normal(85, 5, 8)
fraud_y = np.random.normal(85, 5, 8)

ax.scatter(normal_x, normal_y, c=C['input'], s=20, alpha=0.3, label='정상 거래')
ax.scatter(fraud_x, fraud_y, c=C['sum'], s=80, marker='X',
           edgecolors='white', lw=1, label='사기 거래 (!)')
ax.set_title('③ 이상치 = 핵심 정보 (사기 탐지)', color=C['accent'],
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, facecolor=C['bg'], edgecolor=C['muted'], labelcolor=C['dim'])
ax.text(0.5, 0.05, '이 경우 이상치를 제거하면 안 된다!',
        transform=ax.transAxes, ha='center', color=C['accent'],
        fontsize=10, fontweight='bold')
ax.tick_params(colors=C['muted'])
for spine in ax.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

plt.suptitle('이상치(Outlier)의 두 얼굴 — 제거할 것인가, 보존할 것인가?',
             color=C['text'], fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n📊 이상치가 회귀선에 미친 영향:")
print(f"   원래: y = {m:.2f}x + {b_val:.1f}")
print(f"   이상치 후: y = {m2:.2f}x + {b2:.1f}")
print(f"   기울기 변화: {m:.2f} → {m2:.2f} ({(m2-m)/m*100:.1f}% 변화!)")
print(f"\n🔗 AI 연결:")
print(f"   이상 탐지(Anomaly Detection): Autoencoder, Isolation Forest, One-Class SVM")
print(f"   → 사이버 보안, 제조업 불량 탐지, 의료 진단에서 핵심 AI 기술")
print(f"   → 이상치를 '제거'하는 것이 아니라 '찾는' 것이 목표인 AI 분야가 존재한다")


---

## 4. 상관관계(Correlation) — "특성 선택의 첫 단추"

### 왜 상관 히트맵을 그리는가?

단순히 "어떤 변수가 관련 있는지 보려고"가 아니다. **세 가지 실전 목적**이 있다:

| 목적 | 방법 | 결과 |
|---|---|---|
| **특성 선택**(Feature Selection) | 타겟과 상관 높은 특성 선택 | 모델 성능 향상 |
| **다중공선성**(Multicollinearity) 탐지 | 특성 간 상관 > 0.9 발견 | 하나 제거 → 과적합 방지 |
| **데이터 누수**(Data Leakage) 탐지 | 타겟과 상관 ≈ 1.0 발견 | 미래 정보 누출 차단 |

> **🔗 AI 연결**: 특성이 수천 개인 유전체(Genomics) 데이터나 NLP 임베딩에서는 **차원의 저주**(Curse of Dimensionality) 때문에 특성 선택이 필수이다. PCA, 오토인코더(Autoencoder) 같은 차원 축소 기법의 출발점이 상관 분석이다.


In [ ]:
# ═══════════════════════════════════════════════════
# 상관관계의 3가지 실전 활용 시각화
# ═══════════════════════════════════════════════════

np.random.seed(42)
n = 300

# 특성 생성
feature1 = np.random.normal(50, 10, n)
feature2 = 0.8 * feature1 + np.random.normal(0, 5, n)  # feature1과 높은 상관
feature3 = np.random.normal(30, 8, n)  # 독립
noise = np.random.normal(0, 3, n)
target = 0.6 * feature1 + 0.3 * feature3 + noise  # 진짜 관계
leaky_feature = target * 0.95 + np.random.normal(0, 1, n)  # 데이터 누수!

df_corr = pd.DataFrame({
    'feature_1': feature1,
    'feature_2': feature2,
    'feature_3': feature3,
    'leaky_feat': leaky_feature,
    'target': target,
})

corr = df_corr.corr()

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=C['bg'])

# ── 전체 히트맵 ──
ax = axes[0]
ax.set_facecolor(C['card'])
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('① 상관 히트맵 (전체)', color=C['text'], fontsize=13, fontweight='bold')

# ── 타겟과의 상관 (특성 선택) ──
ax = axes[1]
ax.set_facecolor(C['card'])
target_corr = corr['target'].drop('target').sort_values()
colors_bar = [C['sum'] if abs(v) > 0.8 else C['weight'] if abs(v) > 0.4 else C['input']
              for v in target_corr.values]
bars = ax.barh(target_corr.index, target_corr.values, color=colors_bar, alpha=0.7,
               edgecolor='white', lw=0.5)
ax.axvline(x=0.8, color=C['sum'], ls='--', lw=1, alpha=0.5)
ax.axvline(x=-0.8, color=C['sum'], ls='--', lw=1, alpha=0.5)
ax.set_title('② 타겟과의 상관 (특성 선택)', color=C['text'], fontsize=13, fontweight='bold')
ax.text(0.95, 0.05, '🚨 leaky_feat ≈ 0.99\n→ 데이터 누수 의심!',
        transform=ax.transAxes, ha='right', va='bottom',
        color=C['sum'], fontsize=10, fontweight='bold',
        bbox=dict(facecolor=C['bg'], edgecolor=C['sum'],
                 boxstyle='round,pad=0.3'))
ax.tick_params(colors=C['muted'], labelsize=9)
for spine in ax.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

# ── 특성 간 상관 (다중공선성) ──
ax = axes[2]
ax.set_facecolor(C['card'])
ax.scatter(feature1, feature2, c=C['weight'], s=20, alpha=0.3)
ax.set_xlabel('feature_1', color=C['dim'])
ax.set_ylabel('feature_2', color=C['dim'])
r = np.corrcoef(feature1, feature2)[0, 1]
ax.set_title(f'③ 다중공선성: r={r:.2f}', color=C['weight'],
             fontsize=13, fontweight='bold')
ax.text(0.05, 0.92, f'feature_1과 feature_2는\n거의 같은 정보를 담고 있다.\n→ 하나를 제거해도 된다!',
        transform=ax.transAxes, ha='left', va='top',
        color=C['weight'], fontsize=9,
        bbox=dict(facecolor=C['bg'], edgecolor=C['weight'],
                 boxstyle='round,pad=0.3'))
ax.tick_params(colors=C['muted'], labelsize=8)
for spine in ax.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

plt.tight_layout()
plt.show()

print("\n📊 발견된 문제:")
print(f"   1. 데이터 누수: leaky_feat ↔ target 상관 = {corr.loc['leaky_feat','target']:.3f}")
print(f"      → 이 특성은 모델에 넣으면 안 된다! (미래 정보 포함)")
print(f"   2. 다중공선성: feature_1 ↔ feature_2 상관 = {corr.loc['feature_1','feature_2']:.3f}")
print(f"      → 둘 중 하나를 제거해도 정보 손실이 적다")
print(f"\n🔗 AI 연결:")
print(f"   데이터 누수(Data Leakage)는 ML 실무에서 가장 위험한 실수이다.")
print(f"   Kaggle 경쟁에서도 상위 솔루션이 데이터 누수로 무효화된 사례가 다수 있다.")
print(f"   → 상관 히트맵은 '모델 학습 전 필수 점검 도구'이다.")


---

## 5. EDA는 "선택"이 아니라 "필수" — ML 파이프라인의 관문

### ML 워크플로우에서 EDA의 위치

```
데이터 수집 → 【EDA】 → 전처리 → 특성 공학 → 모델 학습 → 평가 → 배포
              ↑                                                    ↓
              └──────────── 문제 발견 시 되돌아감 ───────────────────┘
```

EDA 없이 모델을 학습하면 **"눈 감고 운전하는 것"**과 같다.

### EDA에서 반드시 확인해야 하는 5가지

| # | 확인 항목 | 도구 | 발견하는 문제 |
|:---:|---|---|---|
| 1 | **결측값** 비율 | `df.isnull().sum()` | 삭제 vs 대체 결정 |
| 2 | **분포** 형태 | 히스토그램 + KDE | 왜도, 이봉분포, 스케일링 필요성 |
| 3 | **이상치** 존재 | 박스플롯 | 제거 vs 보존 결정 |
| 4 | **클래스 불균형** | 카운트플롯 | 오버샘플링/언더샘플링 전략 |
| 5 | **특성 간 관계** | 상관 히트맵 | 특성 선택, 다중공선성, 데이터 누수 |

> **🔗 AI 연결**: Google의 AutoML, H2O.ai 같은 **자동 머신러닝** 도구도 내부적으로 EDA를 수행한다. "자동화된 AI"도 EDA를 건너뛸 수 없다.


In [ ]:
# ═══════════════════════════════════════════════════
# EDA 5단계 체크리스트 — 한 번에 시각화
# ═══════════════════════════════════════════════════

# 문제가 있는 데이터 생성
np.random.seed(42)
n = 500

df_eda = pd.DataFrame({
    'age': np.concatenate([np.random.normal(35, 10, n-5),
                           [150, -3, 200, np.nan, np.nan]]),
    'income': np.concatenate([np.random.exponential(30000, n-3),
                              [np.nan, np.nan, np.nan]]),
    'education': np.random.choice(['고졸', '대졸', '석사', '박사'], n,
                                   p=[0.4, 0.35, 0.2, 0.05]),
    'purchased': np.concatenate([np.zeros(420), np.ones(80)]),  # 불균형!
})

fig, axes = plt.subplots(2, 3, figsize=(16, 10), facecolor=C['bg'])

# ── 1. 결측값 ──
ax = axes[0, 0]
ax.set_facecolor(C['card'])
missing = df_eda.isnull().sum()
colors_miss = [C['sum'] if v > 0 else C['act'] for v in missing.values]
bars = ax.bar(missing.index, missing.values, color=colors_miss, alpha=0.7,
              edgecolor='white', lw=0.5)
for bar, v in zip(bars, missing.values):
    if v > 0:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{v}개', ha='center', color=C['sum'], fontsize=10, fontweight='bold')
ax.set_title('① 결측값 확인', color=C['sum'] if missing.sum() > 0 else C['act'],
             fontsize=13, fontweight='bold')
ax.tick_params(colors=C['muted'], labelsize=9)
for spine in ax.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

# ── 2. 분포 확인 ──
ax = axes[0, 1]
ax.set_facecolor(C['card'])
income_clean = df_eda['income'].dropna()
ax.hist(income_clean, bins=40, color=C['weight'], alpha=0.5,
        edgecolor='white', lw=0.5, density=True)
ax.axvline(income_clean.mean(), color=C['sum'], lw=2, ls='--',
           label=f'평균={income_clean.mean():,.0f}')
ax.axvline(income_clean.median(), color=C['accent'], lw=2, ls=':',
           label=f'중앙값={income_clean.median():,.0f}')
from scipy.stats import skew
sk = skew(income_clean)
ax.set_title(f'② 분포 확인 (왜도={sk:.1f})', color=C['weight'],
             fontsize=13, fontweight='bold')
ax.legend(fontsize=8, facecolor=C['bg'], edgecolor=C['muted'], labelcolor=C['dim'])
ax.tick_params(colors=C['muted'], labelsize=8)
for spine in ax.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

# ── 3. 이상치 확인 ──
ax = axes[0, 2]
ax.set_facecolor(C['card'])
age_clean = df_eda['age'].dropna()
bp = ax.boxplot([age_clean], vert=True, patch_artist=True,
                boxprops=dict(facecolor=C['input']+'33', edgecolor=C['input']),
                medianprops=dict(color=C['accent'], lw=2),
                whiskerprops=dict(color=C['input']),
                capprops=dict(color=C['input']),
                flierprops=dict(marker='o', markerfacecolor=C['sum'],
                               markeredgecolor=C['sum'], markersize=6))
ax.set_title('③ 이상치 확인 (age)', color=C['input'],
             fontsize=13, fontweight='bold')
ax.text(0.95, 0.92, f'min={age_clean.min():.0f}, max={age_clean.max():.0f}\n'
        f'→ -3, 150, 200은 오류!',
        transform=ax.transAxes, ha='right', va='top',
        color=C['sum'], fontsize=9, fontweight='bold',
        bbox=dict(facecolor=C['bg'], edgecolor=C['sum'],
                 boxstyle='round,pad=0.3'))
ax.tick_params(colors=C['muted'], labelsize=8)
for spine in ax.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

# ── 4. 클래스 불균형 ──
ax = axes[1, 0]
ax.set_facecolor(C['card'])
class_counts = df_eda['purchased'].value_counts()
bar_colors = [C['input'], C['sum']]
bars = ax.bar(['미구매 (0)', '구매 (1)'], class_counts.values,
              color=bar_colors, alpha=0.7, edgecolor='white', lw=0.5)
for bar, v in zip(bars, class_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            f'{v}개 ({v/n*100:.0f}%)', ha='center', color=C['text'],
            fontsize=10, fontweight='bold')
ax.set_title('④ 클래스 불균형 확인', color=C['sum'],
             fontsize=13, fontweight='bold')
ax.text(0.95, 0.92, '구매 16% vs 미구매 84%\n→ 불균형 심각!\n→ SMOTE 등 대책 필요',
        transform=ax.transAxes, ha='right', va='top',
        color=C['sum'], fontsize=9, fontweight='bold',
        bbox=dict(facecolor=C['bg'], edgecolor=C['sum'],
                 boxstyle='round,pad=0.3'))
ax.tick_params(colors=C['muted'], labelsize=9)
for spine in ax.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

# ── 5. 상관관계 ──
ax = axes[1, 1]
ax.set_facecolor(C['card'])
numeric_df = df_eda[['age', 'income', 'purchased']].dropna()
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('⑤ 상관관계 확인', color=C['accent'],
             fontsize=13, fontweight='bold')

# ── 종합 판정 ──
ax = axes[1, 2]
ax.set_facecolor(C['card'])
ax.axis('off')
ax.set_title('🔍 EDA 종합 판정', color=C['accent'],
             fontsize=13, fontweight='bold')

checks = [
    ('결측값', '⚠️', f'{missing.sum()}개 발견', C['sum']),
    ('분포', '⚠️', f'income 왜도={sk:.1f} → 로그 변환 필요', C['weight']),
    ('이상치', '⚠️', 'age에 -3, 150, 200 → 제거', C['sum']),
    ('클래스 불균형', '⚠️', '16:84 → SMOTE 적용', C['sum']),
    ('상관관계', '✅', '다중공선성 없음', C['act']),
]

for i, (name, icon, desc, color) in enumerate(checks):
    y = 0.85 - i * 0.17
    ax.text(0.05, y, f'{icon} {name}', transform=ax.transAxes,
            color=color, fontsize=11, fontweight='bold')
    ax.text(0.05, y-0.07, f'   {desc}', transform=ax.transAxes,
            color=C['dim'], fontsize=9)

ax.text(0.05, 0.05, '→ 전처리 후 모델 학습 가능',
        transform=ax.transAxes, color=C['accent'], fontsize=11,
        fontweight='bold')

plt.suptitle('EDA 5단계 체크리스트 — 모델 학습 전 필수 점검',
             color=C['text'], fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n🔗 AI 연결:")
print("   이 5단계는 Kaggle 상위 1% 솔루션의 공통 패턴이다.")
print("   Google AutoML도 내부적으로 결측값 → 분포 → 이상치 → 불균형 → 상관 순서로 점검한다.")
print("   → EDA는 '선택'이 아니라 ML 파이프라인의 '필수 관문'이다.")


---

## 6. 시각화의 미래 — 설명 가능한 AI(Explainable AI, XAI)

### 모델 학습 "후"에도 시각화가 필요하다

EDA는 학습 "전"의 시각화이다. 학습 "후"에는 **모델이 왜 그런 예측을 했는지** 시각화해야 한다.

| 시각화 도구 | 시점 | 목적 |
|---|---|---|
| 히스토그램, 박스플롯, 히트맵 | 학습 **전** (EDA) | 데이터 품질 점검 |
| 학습 곡선(Learning Curve) | 학습 **중** | 과적합/과소적합 진단 |
| **SHAP**, **LIME**, Grad-CAM | 학습 **후** (XAI) | 모델 예측 근거 설명 |

> **🔗 AI 연결**: EU의 **AI Act**(2024)는 고위험 AI 시스템에 대해 "설명 가능성"을 법적으로 요구한다. 의료, 금융, 법률 AI는 **왜 그런 판단을 했는지 설명할 수 있어야** 한다. 시각화는 이 설명의 핵심 도구이다.

### 맛보기: 특성 중요도(Feature Importance) 시각화

이것은 7주차(결정 트리)에서 본격적으로 다루지만, 미리 맛만 보자.


In [ ]:
# ═══════════════════════════════════════════════════
# 특성 중요도 시각화 (XAI 맛보기)
# ═══════════════════════════════════════════════════
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

# Iris 데이터
iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
y_iris = iris.target

# 모델 학습
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_iris, y_iris)

# 특성 중요도
importances = pd.Series(rf.feature_importances_,
                         index=iris.feature_names).sort_values()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor=C['bg'])

# ── 특성 중요도 ──
ax1.set_facecolor(C['card'])
colors_imp = [C['input'] if v < 0.2 else C['weight'] if v < 0.4 else C['act']
              for v in importances.values]
ax1.barh(importances.index, importances.values, color=colors_imp,
         alpha=0.7, edgecolor='white', lw=0.5)
ax1.set_title('특성 중요도(Feature Importance)', color=C['text'],
              fontsize=14, fontweight='bold')
ax1.set_xlabel('중요도', color=C['dim'])
for i, (name, val) in enumerate(importances.items()):
    ax1.text(val + 0.01, i, f'{val:.3f}', va='center',
             color=C['text'], fontsize=10, fontfamily='monospace')
ax1.tick_params(colors=C['muted'], labelsize=9)
for spine in ax1.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

# ── ML 파이프라인 위치도 ──
ax2.set_facecolor(C['card'])
ax2.axis('off')
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 6)
ax2.set_title('시각화가 쓰이는 3가지 시점', color=C['accent'],
              fontsize=14, fontweight='bold')

stages = [
    (1.5, 4.5, '학습 전\nEDA', C['input'],
     '히스토그램\n박스플롯\n히트맵'),
    (5, 4.5, '학습 중\nMonitoring', C['weight'],
     '학습 곡선\n손실 그래프\n검증 정확도'),
    (8.5, 4.5, '학습 후\nXAI', C['act'],
     '특성 중요도\nSHAP\nGrad-CAM'),
]

for x, y, title, color, tools in stages:
    box = FancyBboxPatch((x-1.2, y-0.8), 2.4, 1.6,
                          boxstyle="round,pad=0.15",
                          facecolor=color+'22', edgecolor=color, lw=2)
    ax2.add_patch(box)
    ax2.text(x, y+0.3, title, ha='center', va='center',
             color=color, fontsize=11, fontweight='bold')
    ax2.text(x, y-0.3, tools, ha='center', va='center',
             color=C['dim'], fontsize=8)

# 화살표
for i in range(2):
    ax2.annotate('', xy=(stages[i+1][0]-1.2, 4.5),
                xytext=(stages[i][0]+1.2, 4.5),
                arrowprops=dict(arrowstyle='->', color=C['muted'], lw=2))

# 5주차 표시
ax2.text(1.5, 2.5, '← 5주차: 여기!', ha='center', color=C['input'],
         fontsize=12, fontweight='bold')
ax2.text(5, 2.5, '← 9~10주차', ha='center', color=C['weight'],
         fontsize=10)
ax2.text(8.5, 2.5, '← 13~14주차', ha='center', color=C['act'],
         fontsize=10)

plt.tight_layout()
plt.show()

print("\n💡 지금 배우는 EDA 시각화 기술은 ML 파이프라인의 첫 번째 관문이다.")
print("   이것이 9주차(학습 곡선), 13주차(SHAP/Grad-CAM)로 이어진다.")
print("\n🔗 AI 연결:")
print("   EU AI Act(2024): 고위험 AI는 '왜 그런 판단을 했는가?'를 설명해야 한다")
print("   → 시각화 = AI의 '설명 능력'을 만드는 핵심 기술")


---

## 7. 고차원 데이터 시각화 — PCA와 t-SNE 맛보기

우리 눈은 2D/3D만 볼 수 있다. 하지만 실제 데이터는 수십~수백 차원이다.

**차원 축소**(Dimensionality Reduction)로 고차원 데이터를 2D로 "압축"하여 시각화한다.

| 기법 | 원리 | 용도 |
|---|---|---|
| **PCA** | 분산이 최대인 축으로 투영 | 전체 구조 파악, 빠름 |
| **t-SNE** | 이웃 관계를 보존하며 축소 | 군집(Cluster) 시각화, 느림 |

> **🔗 AI 연결**: GPT의 토큰 임베딩(768~16384차원)을 t-SNE로 시각화하면, 의미가 비슷한 단어들이 가까이 모여 있는 것을 확인할 수 있다.


In [ ]:
# ═══════════════════════════════════════════════════
# PCA & t-SNE — 고차원 데이터를 2D로 시각화
# ═══════════════════════════════════════════════════
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

# 손글씨 숫자 데이터 (64차원 → 2차원)
digits = load_digits()
X_digits = StandardScaler().fit_transform(digits.data)
y_digits = digits.target

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), facecolor=C['bg'])

# ── PCA ──
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_digits)

ax1.set_facecolor(C['card'])
scatter1 = ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=y_digits,
                        cmap='tab10', s=15, alpha=0.6)
ax1.set_title(f'PCA (64차원 → 2차원)\n설명 분산: {pca.explained_variance_ratio_.sum():.1%}',
              color=C['text'], fontsize=14, fontweight='bold')
ax1.set_xlabel('PC1', color=C['dim'])
ax1.set_ylabel('PC2', color=C['dim'])
plt.colorbar(scatter1, ax=ax1, label='숫자')
ax1.tick_params(colors=C['muted'])
for spine in ax1.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

# ── t-SNE ──
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_digits)

ax2.set_facecolor(C['card'])
scatter2 = ax2.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_digits,
                        cmap='tab10', s=15, alpha=0.6)
ax2.set_title('t-SNE (64차원 → 2차원)\n군집이 명확히 분리된다!',
              color=C['text'], fontsize=14, fontweight='bold')
ax2.set_xlabel('t-SNE 1', color=C['dim'])
ax2.set_ylabel('t-SNE 2', color=C['dim'])
plt.colorbar(scatter2, ax=ax2, label='숫자')
ax2.tick_params(colors=C['muted'])
for spine in ax2.spines.values():
    spine.set_color(C['muted'])
    spine.set_alpha(0.3)

plt.suptitle('고차원 데이터 시각화 — 64차원 손글씨 숫자를 2차원으로 압축',
             color=C['text'], fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n📊 데이터: 손글씨 숫자 0~9 (8×8 이미지 = 64차원)")
print(f"   PCA: 전체 구조 파악에 유리, 설명 분산 {pca.explained_variance_ratio_.sum():.1%}")
print(f"   t-SNE: 군집 분리에 탁월, 같은 숫자가 한 곳에 모인다")
print(f"\n🔗 AI 연결:")
print(f"   GPT의 단어 임베딩을 t-SNE로 시각화하면:")
print(f"   '왕'-'남자'+'여자' ≈ '여왕'  ← 이런 관계가 2D에서 눈에 보인다")
print(f"   Word2Vec, BERT 임베딩의 품질을 점검하는 표준 방법이 t-SNE 시각화이다")


---

## 🎯 핵심 정리 — 5주차에서 배운 "왜?"

| 개념 | 도구 수준 (기존) | 전문가 수준 (보강) | 최신 AI 연결 |
|---|---|---|---|
| **시각화** | "그래프를 그린다" | 숫자는 거짓말한다 — 시각적 확인 필수 | 학습 곡선, SHAP, Grad-CAM |
| **분포** | "히스토그램을 본다" | 정규분포 가정 → 왜도 확인 → 변환 | Layer Normalization |
| **이상치** | "박스플롯으로 찾는다" | 오류 vs 희소 사건 구분 | 이상 탐지(Anomaly Detection) AI |
| **상관관계** | "히트맵을 그린다" | 특성 선택 + 다중공선성 + 데이터 누수 탐지 | 차원의 저주, PCA, 오토인코더 |
| **EDA** | "데이터를 탐색한다" | ML 파이프라인의 필수 관문 | AutoML도 EDA부터 시작 |
| **XAI** | (다루지 않음) | 모델 예측 근거 시각화 | EU AI Act 법적 요구사항 |
| **차원 축소** | (다루지 않음) | PCA/t-SNE로 고차원 시각화 | 임베딩 품질 점검 |

### 핵심 메시지

> *"시각화는 '예쁜 그래프 만들기'가 아니다.*
> *데이터의 진실을 발견하고, 모델의 판단을 설명하는 기술이다.*
> *EDA 없이 모델을 학습하는 것은 눈 감고 운전하는 것과 같다.*
> *EU는 이미 AI에게 '설명하라'고 법으로 요구하고 있다."*
